# 03 - Deep Agent com planejamento e ferramentas

O notebook anterior criou um plano antes de responder, mas sem dados externos. Aqui o agente planeja e depois chama **ferramentas** para resolver a tarefa.

Caso prático: responder qual produto mais vendeu no primeiro trimestre. O agente:

1. cria um plano com `write_todos`;
2. consulta as vendas de cada mês com a ferramenta `consultar_vendas`;
3. faz as contas com a ferramenta `calcular`;
4. responde com o resultado.

Fluxo:

```
START → planeja (write_todos) → ferramentas (consultar_vendas, calcular) → resposta final → END
```

Tudo fica registrado no MLflow: cada execução vira um trace, com o plano e as chamadas de ferramenta como spans.

## 1. Configuração

A chave e o modelo vêm do `.env` na raiz do projeto, como nos outros notebooks.

O rastreamento usa o MLflow com um banco SQLite, `mlflow.db`, criado na pasta deste notebook. O `mlflow.langchain.autolog()` registra cada execução, com as chamadas ao modelo, ao plano e às ferramentas.

In [1]:
import ast
import json
import operator
import os
import unicodedata

import mlflow
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from langchain.agents.middleware import TodoListMiddleware
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from deepagents import create_deep_agent

load_dotenv(find_dotenv())

mlflow.set_tracking_uri("sqlite:///mlflow.db")
EXPERIMENTO = "deep-agent-ferramentas"
experimento = mlflow.set_experiment(EXPERIMENTO)
mlflow.langchain.autolog()

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-5-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
)

2026/11/17 07:38:44 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at /home/leonardo/Documentos/github/loop_engineering_study/.venv/lib/python3.13/site-packages/mlflow/assistant/skills/mlflow-trace-instrumentation/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


2026/11/17 07:38:46 INFO mlflow.tracking.fluent: Experiment with name 'deep-agent-ferramentas' does not exist. Creating a new experiment.


## 2. Ferramentas

- `consultar_vendas`: devolve as vendas de um mês. Os dados são simulados, para o notebook rodar sem API externa.
- `calcular`: faz contas com números e as quatro operações. Usa `ast`, como no notebook 04 do LangGraph. Nunca use `eval` com texto gerado por LLM.

In [2]:
VENDAS_API = {
    "janeiro": {"cafe": 120, "pao": 300, "queijo": 45},
    "fevereiro": {"cafe": 135, "pao": 280, "queijo": 50},
    "marco": {"cafe": 150, "pao": 310, "queijo": 38},
}


@tool
def consultar_vendas(mes: str) -> str:
    """Consulta as vendas de um mês do primeiro trimestre. Retorna a quantidade vendida de cada produto em JSON. Meses válidos: janeiro, fevereiro, março."""
    chave = unicodedata.normalize("NFKD", mes.lower().strip()).encode("ascii", "ignore").decode()
    dados = VENDAS_API.get(chave)
    if dados is None:
        return json.dumps({"mes": mes, "erro": "mês não encontrado"}, ensure_ascii=False)
    return json.dumps({"mes": mes, "vendas": dados}, ensure_ascii=False)


OPERACOES = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
}


def _avaliar(no: ast.AST) -> float:
    if isinstance(no, ast.Constant) and isinstance(no.value, (int, float)):
        return no.value
    if isinstance(no, ast.BinOp) and type(no.op) in OPERACOES:
        return OPERACOES[type(no.op)](_avaliar(no.left), _avaliar(no.right))
    if isinstance(no, ast.UnaryOp) and isinstance(no.op, ast.USub):
        return -_avaliar(no.operand)
    raise ValueError("expressão não permitida")


@tool
def calcular(expressao: str) -> str:
    """Calcula uma expressão aritmética com números, + - * / e parênteses. Exemplo: '(120 + 135) * 2'."""
    return str(round(_avaliar(ast.parse(expressao, mode="eval").body), 2))


FERRAMENTAS = [consultar_vendas, calcular]

print(consultar_vendas.name, "|", calcular.name)

consultar_vendas | calcular


## 3. Criando o agente

O `TodoListMiddleware` adiciona o `write_todos`. O `system_prompt` pede a ordem: plano primeiro, depois as ferramentas e a resposta final.

In [3]:
agente = create_deep_agent(
    model=llm,
    tools=FERRAMENTAS,
    middleware=[TodoListMiddleware()],
    system_prompt=(
        "Você é um analista de vendas. Em tarefas com etapas, a primeira ação é chamar write_todos "
        "com um plano de 3 a 5 passos. Depois, use as ferramentas para buscar os dados e fazer as contas. "
        "Ao final, responda em português, com os números usados."
    ),
)

## 4. Executando

A função `executar` roda a tarefa e mostra as mensagens na ordem em que aconteceram:

- `Plano` é a lista de passos da primeira chamada de `write_todos`. Chamadas seguintes atualizam o status.
- `→` é uma chamada de ferramenta.
- `←` é o resultado devolvido pela ferramenta.
- `Resposta` é a mensagem final.

In [4]:
def resumir(texto: str, limite: int = 200) -> str:
    return texto if len(texto) <= limite else texto[:limite] + "..."


def executar(tarefa: str) -> dict:
    estado = agente.invoke({"messages": [HumanMessage(content=tarefa)]})

    linhas = [f"**Tarefa:** {tarefa}", ""]
    plano_mostrado = False
    for mensagem in estado["messages"][1:]:
        if isinstance(mensagem, AIMessage) and mensagem.tool_calls:
            for chamada in mensagem.tool_calls:
                if chamada["name"] == "write_todos" and not plano_mostrado:
                    linhas.append("**Plano:**")
                    for item in chamada["args"].get("todos", []):
                        linhas.append(f"- {item.get('content', item)}")
                    linhas.append("")
                    plano_mostrado = True
                elif chamada["name"] == "write_todos":
                    linhas.append("→ `write_todos` (atualiza o status dos passos)")
                else:
                    linhas.append(f"→ `{chamada['name']}({resumir(str(chamada['args']))})`")
        elif isinstance(mensagem, ToolMessage) and mensagem.name != "write_todos":
            linhas.append(f"← `{resumir(str(mensagem.content))}`")
        elif isinstance(mensagem, AIMessage) and mensagem.content:
            linhas.append("")
            linhas.append(f"**Resposta:** {mensagem.content}")
    display(Markdown("\n".join(linhas)))
    return estado


estado = executar(
    "Qual produto mais vendeu no primeiro trimestre? Mostre o total de cada produto no trimestre."
)

**Tarefa:** Qual produto mais vendeu no primeiro trimestre? Mostre o total de cada produto no trimestre.

**Plano:**
- Coletar vendas de janeiro, fevereiro e março
- Somar totais por produto no trimestre
- Identificar produto com maior venda no trimestre
- Apresentar resultado com total de cada produto e indicar o mais vendido

→ `consultar_vendas({'mes': 'janeiro'})`
← `{"mes": "janeiro", "vendas": {"cafe": 120, "pao": 300, "queijo": 45}}`
→ `write_todos` (atualiza o status dos passos)
→ `consultar_vendas({'mes': 'fevereiro'})`
← `{"mes": "fevereiro", "vendas": {"cafe": 135, "pao": 280, "queijo": 50}}`
→ `write_todos` (atualiza o status dos passos)
→ `consultar_vendas({'mes': 'marco'})`
← `{"mes": "marco", "vendas": {"cafe": 150, "pao": 310, "queijo": 38}}`
→ `calcular({'expressao': '(120 + 135 + 150)'})`
← `405`
→ `calcular({'expressao': '(300 + 280 + 310)'})`
← `890`
→ `calcular({'expressao': '(45 + 50 + 38)'})`
← `133`
→ `write_todos` (atualiza o status dos passos)

**Resposta:** Resultado do primeiro trimestre (janeiro + fevereiro + março):

- café: 120 + 135 + 150 = 405
- pão: 300 + 280 + 310 = 890
- queijo: 45 + 50 + 38 = 133

Produto mais vendido no trimestre: pão, com 890 unidades.

Números usados:
Janeiro — café 120, pão 300, queijo 45
Fevereiro — café 135, pão 280, queijo 50
Março — café 150, pão 310, queijo 38
Totais — café 405, pão 890, queijo 133

## 5. Conferindo a ordem

Listamos as chamadas em ordem. O esperado é `write_todos` primeiro, depois as chamadas de `consultar_vendas` e, por fim, `calcular` se o agente usou.

In [5]:
ordem = []
for mensagem in estado["messages"]:
    if isinstance(mensagem, AIMessage):
        ordem.extend(chamada["name"] for chamada in mensagem.tool_calls)

print(" → ".join(ordem) if ordem else "nenhuma ferramenta chamada")
print("Planejou primeiro:", bool(ordem) and ordem[0] == "write_todos")
print("Usou ferramentas de dados:", any(nome == "consultar_vendas" for nome in ordem))

write_todos → consultar_vendas → write_todos → consultar_vendas → write_todos → consultar_vendas → calcular → calcular → calcular → write_todos
Planejou primeiro: True
Usou ferramentas de dados: True


## 6. Acompanhando o rastreamento no MLflow

Cada execução virou um trace no experimento `deep-agent-ferramentas`. Para ver:

1. Abra um terminal na pasta `notebooks/04-DeepAgents` (o caminho do `mlflow.db` é relativo a ela).
2. Rode:

```bash
uv run --project ../.. mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```

3. Abra `http://127.0.0.1:5000` no navegador.
4. Em **Experiments**, escolha `deep-agent-ferramentas` e abra a aba **Traces**.
5. Clique no trace mais recente. Os spans mostram, em ordem: o plano (`write_todos`), cada chamada de `consultar_vendas` com a entrada e a saída, e a resposta final do modelo.
6. Pressione `Ctrl+C` no terminal para parar a interface.

A célula seguinte lista os spans pelo notebook, sem abrir a interface.

In [6]:
traces = mlflow.search_traces(experiment_ids=[experimento.experiment_id], max_results=10)
print(f"Traces registrados no experimento: {len(traces)}")

trace = mlflow.get_trace(traces.iloc[0]["trace_id"])
print(f"\nSpans do trace mais recente ({trace.info.trace_id}):")
for span in trace.data.spans:
    if span.span_type in ("TOOL", "CHAT_MODEL"):
        print(f"- {span.name} ({span.span_type})")

Traces registrados no experimento: 1

Spans do trace mais recente (tr-af079278398f924e4d96f54e94b57202):
- ChatOpenAI (CHAT_MODEL)
- write_todos (TOOL)
- ChatOpenAI (CHAT_MODEL)
- consultar_vendas (TOOL)
- ChatOpenAI (CHAT_MODEL)
- write_todos (TOOL)
- ChatOpenAI (CHAT_MODEL)
- consultar_vendas (TOOL)
- ChatOpenAI (CHAT_MODEL)
- write_todos (TOOL)
- ChatOpenAI (CHAT_MODEL)
- consultar_vendas (TOOL)
- ChatOpenAI (CHAT_MODEL)
- calcular (TOOL)
- ChatOpenAI (CHAT_MODEL)
- calcular (TOOL)
- ChatOpenAI (CHAT_MODEL)
- calcular (TOOL)
- ChatOpenAI (CHAT_MODEL)
- write_todos (TOOL)
- ChatOpenAI (CHAT_MODEL)


/tmp/ipykernel_48040/3747251566.py:1: FutureWarning: Parameter 'experiment_ids' is deprecated. Please use 'locations' instead.
  traces = mlflow.search_traces(experiment_ids=[experimento.experiment_id], max_results=10)


## Observações

- O plano e as ferramentas são controlados pelo `system_prompt`. Por isso a célula de conferência verifica a ordem das chamadas.
- As ferramentas são comuns `@tool`, como no notebook 04 do LangGraph. O deep agent não muda a forma de definir ferramentas.
- O rastreamento é automático pelo `mlflow.langchain.autolog()`. Os spans `TOOL` mostram a entrada e a saída de cada ferramenta.
- O resultado pode variar entre execuções. Compare o plano e as chamadas com a resposta final.